# Classification — PyCaret 4.0

Predict a binary / multi-class label. This notebook shows the canonical
4.0 OOP pattern end-to-end:

1. Load data
2. Construct and fit a `ClassificationExperiment`
3. Compare models → get a typed `CompareResult`
4. Tune the best
5. Predict on test / new data
6. Save & load the fitted pipeline

Every verb returns a typed dataclass with fitted pipeline, metrics, and an
event trace. Nothing is implicit / global / stateful.


In [1]:
# PyCaret 4.0 — OOP-only engine
# https://github.com/pycaret/pycaret
import warnings
warnings.filterwarnings("ignore")
import pycaret
print("PyCaret", pycaret.__version__)


PyCaret 4.0.0.dev0


## 1. Load data

In [2]:
from pycaret.datasets import get_data
data = get_data("juice")
data.head()


,Id,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,1,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,2,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,3,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,4,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,5,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0


,Id,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,1,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,2,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,3,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,4,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,5,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0


## 2. Fit an experiment

The `ClassificationExperiment` is a `sklearn.base.BaseEstimator` subclass.
`fit(data)` runs the full preprocessing pipeline and train/test split.

In [3]:
from pycaret.tasks import ClassificationExperiment

exp = ClassificationExperiment(
    target="Purchase",
    session_id=42,
    n_jobs=1,            # set to -1 for all cores
    log_experiment=True, # capture a typed event stream
).fit(data)

print("is_fitted:", exp.__sklearn_is_fitted__())
print("train size:", exp.X_train.shape, "test size:", exp.X_test.shape)


is_fitted: True
train size: (749, 18) test size: (321, 18)


## 3. Compare models

Returns a `CompareResult` dataclass: `.best`, `.models`, `.leaderboard`, `.ranked_ids`, `.events`.

In [4]:
result = exp.compare_models(include=["lr", "dt", "rf"])
print(type(result).__name__)
result.leaderboard


Processing:   0%|          | 0/17 [00:00<?, ?it/s]

Processing:  29%|██▉       | 5/17 [00:01<00:04,  2.89it/s]

Processing:  53%|█████▎    | 9/17 [00:02<00:02,  3.97it/s]

Processing:  65%|██████▍   | 11/17 [00:02<00:01,  5.02it/s]

Processing:  76%|███████▋  | 13/17 [00:05<00:02,  1.86it/s]

Processing:  88%|████████▊ | 15/17 [00:05<00:00,  2.51it/s]

Processing: 100%|██████████| 17/17 [00:05<00:00,  2.97it/s]

                       Model  Accuracy     AUC  Recall   Prec.      F1  \
lr       Logistic Regression    0.6008  0.4633  0.6008  0.4872  0.4728   
rf  Random Forest Classifier    0.5369  0.5068  0.5369  0.5202  0.5240   
dt  Decision Tree Classifier    0.5074  0.4844  0.5074  0.5095  0.5067   

     Kappa     MCC  TT (Sec)  
lr -0.0074 -0.0031     0.171  
rf -0.0092 -0.0088     0.278  
dt -0.0310 -0.0311     0.058  
CompareResult


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.6008,0.4633,0.6008,0.4872,0.4728,-0.0074,-0.0031,0.171
rf,Random Forest Classifier,0.5369,0.5068,0.5369,0.5202,0.5240,-0.0092,-0.0088,0.278
dt,Decision Tree Classifier,0.5074,0.4844,0.5074,0.5095,0.5067,-0.0310,-0.0311,0.058


In [5]:
best = result.best
type(best).__name__


'LogisticRegression'

## 4. Tune the best model

Returns a `TuneResult` with `.pipeline`, `.best_params`, `.cv_results`.

In [6]:
tuned_result = exp.tune_model(best, n_iter=5)
tuned = tuned_result.pipeline
tuned_result.cv_results


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 5 candidates, totalling 50 fits


Processing:  43%|████▎     | 3/7 [00:06<00:09,  2.32s/it]

Processing:  86%|████████▌ | 6/7 [00:08<00:01,  1.31s/it]

Processing: 100%|██████████| 7/7 [00:08<00:00,  1.07s/it]

Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
      Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                          
0       0.6133  0.5285  0.6133  0.3762  0.4663  0.0000  0.0000
1       0.6000  0.3913  0.6000  0.5591  0.5427  0.0474  0.0578
2       0.6267  0.4250  0.6267  0.7679  0.4960  0.0420  0.1464
3       0.6000  0.5517  0.6000  0.5037  0.4816 -0.0108 -0.0224
4       0.6000  0.5532  0.6000  0.5037  0.4816 -0.0108 -0.0224
5       0.5333  0.4093  0.5333  0.4359  0.4596 -0.1194 -0.1503
6       0.6133  0.4655  0.6133  0.3762  0.4663  0.0000  0.0000
7       0.6267  0.4467  0.6267  0.7699  0.5076  0.0789  0.2027
8       0.5867  0.3763  0.5867  0.3568  0.4437 -0.0265 -0.0949
9       0.6081  0.5670  0.6081  0.5715  0.5192  0.0428  0.0658
Mean    0.6008  0.4715  0.6008  0.5221  0.4865  0.0044  0.0183
Std     0.0254  0.0690  0.0254  

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6133,0.5285,0.6133,0.3762,0.4663,0.0000,0.0000
1,0.6000,0.3913,0.6000,0.5591,0.5427,0.0474,0.0578
2,0.6267,0.4250,0.6267,0.7679,0.4960,0.0420,0.1464
3,0.6000,0.5517,0.6000,0.5037,0.4816,-0.0108,-0.0224
4,0.6000,0.5532,0.6000,0.5037,0.4816,-0.0108,-0.0224
5,0.5333,0.4093,0.5333,0.4359,0.4596,-0.1194,-0.1503
6,0.6133,0.4655,0.6133,0.3762,0.4663,0.0000,0.0000
7,0.6267,0.4467,0.6267,0.7699,0.5076,0.0789,0.2027
8,0.5867,0.3763,0.5867,0.3568,0.4437,-0.0265,-0.0949


## 5. Predict on the held-out test set

`predict_model` returns a `PredictResult` with `.predictions` (DataFrame) and `.metrics` (DataFrame).

In [7]:
preds = exp.predict_model(tuned)
preds.predictions.head()


                 Model  Accuracy     AUC  Recall   Prec.      F1   Kappa  \
0  Logistic Regression    0.6137  0.4439  0.6137  0.5863  0.4999  0.0303   

      MCC  
0  0.0628  


,Id,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,...,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE,Purchase,prediction_label,prediction_score
426,427,262,4,1.99,2.09,0.0,0.0,0,0,0.104858,...,1.99,0.10,No,0.000000,0.000000,0.10,4,CH,CH,0.5485
590,591,253,7,1.86,2.09,0.1,0.0,0,0,0.584000,...,1.76,0.33,Yes,0.000000,0.053763,0.23,0,CH,CH,0.6909
573,574,255,3,1.99,2.29,0.0,0.0,0,0,0.199771,...,1.99,0.30,No,0.000000,0.000000,0.30,3,CH,MM,0.5111
789,790,275,2,1.96,2.18,0.0,0.8,0,1,0.600000,...,1.96,-0.58,No,0.366972,0.000000,0.22,2,MM,CH,0.7440
966,967,236,1,1.75,1.99,0.0,0.0,0,0,0.548160,...,1.75,0.24,No,0.000000,0.000000,0.24,1,MM,CH,0.7405


## 6. Persist the fitted pipeline

`save_model` / `load_model` are stateless top-level utilities — no Experiment required to load.

In [8]:
from pycaret import save_model, load_model
from pathlib import Path

out = save_model(tuned, Path("artifacts") / "juice_classifier")
print("saved to:", out)

restored = load_model(out)
print("restored:", type(restored).__name__)


saved to: C:\Users\moezs\pycaret\pycaret\artifacts\juice_classifier.pkl
restored: LogisticRegression


## 7. Inspect the event stream

This is what a React UI / LLM agent subscribes to.

In [9]:
events = exp.events
for e in events:
    dur = f"{e.duration_ms:7.1f} ms" if e.duration_ms else " " * 10
    print(f"{e.kind.value:30s} {dur}  {e.message}")


experiment.started                         Starting classification experiment
experiment.fitted                847.8 ms  Experiment fitted and ready
model.compare.started                      
model.compare.finished          5817.5 ms  
model.tune.started                         
model.tuned                    11154.7 ms  
model.predicted                            
